# Обучение ретриверов

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc
import re
import json
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import faiss
import ir_datasets
from datasets import Dataset
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm
from peft import LoraConfig, get_peft_model, TaskType
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.losses import (
    MultipleNegativesRankingLoss,
    CachedMultipleNegativesRankingLoss,
)
from sentence_transformers.models import Transformer, Pooling

set_seed(42)
print(f"GPUs: {torch.cuda.device_count()}, current: {torch.cuda.get_device_name(0)}")

## Загрузка данных

In [ ]:
with open("data/processed/train_data_final.json", encoding="utf-8") as f:
    train_data = json.load(f)
with open("data/processed/val_data_final.json", encoding="utf-8") as f:
    val_data = json.load(f)

print(f"train={len(train_data)} val={len(val_data)}")

In [ ]:
def make_train_dataset(data, is_e5, use_inst_neg=True):
    qp = "query: " if is_e5 else ""
    dp = "passage: " if is_e5 else ""
    rows = []
    for item in data:
        hard = item["hard_negatives"]
        rand = item.get("random_negatives", [])
        inst_neg = item["instruction_negative"]
        inst_neg = inst_neg["passage"] if isinstance(inst_neg, dict) else inst_neg

        negs_plain = (hard + rand)[:5]
        row1 = {"anchor": qp + item["query"], "positive": dp + item["positive_text"]}
        for i, n in enumerate(negs_plain, 1):
            row1[f"negative_{i}"] = dp + n
        rows.append(row1)

        instr_query = f"{item['generated_instruction']} {item['query']}"
        negs_instr = ([inst_neg] + hard + rand)[:5] if use_inst_neg else (hard + rand)[:5]
        row2 = {"anchor": qp + instr_query, "positive": dp + item["positive_text"]}
        for i, n in enumerate(negs_instr, 1):
            row2[f"negative_{i}"] = dp + n
        rows.append(row2)
    return Dataset.from_list(rows)

## IR Evaluator на mMARCO dev

Метрики на подмножестве запросов с BM25-кандидатами как пулом, подходит для быстрого замера во время экспериментов

In [ ]:
def build_ir_evaluator_mmarco(is_e5, name="val_mmarco", max_q=300, top_bm25=200):
    qp = "query: " if is_e5 else ""
    dp = "passage: " if is_e5 else ""
    ds = ir_datasets.load("mmarco/ru/dev/small")

    queries = {}
    for q in ds.queries_iter():
        queries[str(q.query_id)] = qp + q.text
        if len(queries) >= max_q:
            break
    qids = set(queries.keys())

    relevant_docs = defaultdict(set)
    for qr in ds.qrels_iter():
        if str(qr.query_id) in qids and int(qr.relevance) > 0:
            relevant_docs[str(qr.query_id)].add(str(qr.doc_id))

    cand_per_q = defaultdict(list)
    all_doc_ids = set()
    for sd in ds.scoreddocs_iter():
        qid = str(sd.query_id)
        if qid not in qids:
            continue
        if len(cand_per_q[qid]) >= top_bm25:
            continue
        cand_per_q[qid].append(str(sd.doc_id))
        all_doc_ids.add(str(sd.doc_id))

    # позитивы обязательно кладем
    for rels in relevant_docs.values():
        all_doc_ids.update(rels)

    docs_store = ds.docs_store()
    corpus = {d: dp + docs_store.get(d).text[:2000] for d in all_doc_ids}

    return InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=dict(relevant_docs),
        name=name,
        show_progress_bar=False,
        mrr_at_k=[10],
        ndcg_at_k=[10],
        accuracy_at_k=[1, 5, 10],
        precision_recall_at_k=[10],
        map_at_k=[10],
        batch_size=128,
    )

val_evaluator_e5 = build_ir_evaluator_mmarco(is_e5=True)
val_evaluator_plain = build_ir_evaluator_mmarco(is_e5=False)

## SentenceTransformer

In [ ]:
def load_st_model(model_path):
    meta_path = os.path.join(model_path, "retriever_meta.json")
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        is_e5 = meta.get("is_e5", False)
        is_llm = meta.get("is_llm", False)
    else:
        is_e5 = "e5" in model_path.lower()
        is_llm = any(x in model_path.lower() for x in ["qwen", "llama", "mistral"])

    model = SentenceTransformer(model_path, device="cuda", trust_remote_code=True)

    if is_llm:
        word_emb = model[0]
        if word_emb.tokenizer.pad_token is None:
            word_emb.tokenizer.pad_token = word_emb.tokenizer.eos_token
            word_emb.auto_model.config.pad_token_id = word_emb.tokenizer.pad_token_id
        word_emb.tokenizer.padding_side = "left"
        original_tokenize = word_emb.tokenize
        eos = word_emb.tokenizer.eos_token

        def tokenize_with_eos(texts):
            texts = [t if t.endswith(eos) else t + eos for t in texts]
            return original_tokenize(texts)

        word_emb.tokenize = tokenize_with_eos

    return model, is_e5, is_llm


def build_st_model(model_id, use_lora, lora_rank, lora_alpha, max_seq_length):
    is_llm = any(x in model_id.lower() for x in ["qwen", "llama", "mistral"])

    if is_llm:
        word_emb = Transformer(
            model_id,
            max_seq_length=max_seq_length,
            model_args={"trust_remote_code": True, "torch_dtype": torch.bfloat16},
        )
        if word_emb.tokenizer.pad_token is None:
            word_emb.tokenizer.pad_token = word_emb.tokenizer.eos_token
            word_emb.auto_model.config.pad_token_id = word_emb.tokenizer.pad_token_id
        word_emb.tokenizer.padding_side = "left"
        original_tokenize = word_emb.tokenize
        eos = word_emb.tokenizer.eos_token

        def tokenize_with_eos(texts):
            texts = [t if t.endswith(eos) else t + eos for t in texts]
            return original_tokenize(texts)

        word_emb.tokenize = tokenize_with_eos
        pooling = Pooling(word_emb.get_word_embedding_dimension(), pooling_mode_lasttoken=True)
        model = SentenceTransformer(modules=[word_emb, pooling], device="cuda")
    else:
        model = SentenceTransformer(model_id, device="cuda", trust_remote_code=True)
        model.max_seq_length = max_seq_length
        model[0].max_seq_length = max_seq_length
        model[0].tokenizer.model_max_length = max_seq_length

    if use_lora:
        if is_llm:
            target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                              "down_proj", "up_proj", "gate_proj"]
            model[0].auto_model.config.use_cache = False
            model[0].auto_model.enable_input_require_grads()
        else:
            target_modules = ["query", "value", "key", "dense"]

        peft_config = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=lora_rank,
            lora_alpha=lora_alpha,
            lora_dropout=0.1,
            target_modules=target_modules,
        )
        model[0].auto_model = get_peft_model(model[0].auto_model, peft_config)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        print(f"trainable: {trainable}/{total} ({100 * trainable / total:.4f}%)")

    return model, is_llm

## Обучение

In [ ]:
def run_experiment(model_id, run_name, use_lora=False, lora_rank=16, lora_alpha=32,
                   lr=2e-5, epochs=3, batch_size=64, grad_accum=1,
                   max_seq_length=320, use_cached_loss=False,
                   use_inst_neg=True, train_data_subset=None):
    print(f"\n{'=' * 60}\n{run_name}\n{'=' * 60}")

    model, is_llm = build_st_model(model_id, use_lora, lora_rank, lora_alpha, max_seq_length)

    if use_cached_loss:
        loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=8)
    else:
        loss = MultipleNegativesRankingLoss(model)

    is_e5 = "e5" in model_id.lower()
    data_for_run = train_data_subset if train_data_subset is not None else train_data
    ds = make_train_dataset(data_for_run, is_e5=is_e5, use_inst_neg=use_inst_neg).shuffle(seed=42)
    evaluator = val_evaluator_e5 if is_e5 else val_evaluator_plain

    steps_per_epoch = max(1, len(ds) // (batch_size * max(1, grad_accum)))
    total_steps = max(1, steps_per_epoch * epochs)
    eval_steps_eff = max(20, min(200, total_steps // 4))

    args = SentenceTransformerTrainingArguments(
        output_dir=f"./models/{run_name}",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        learning_rate=lr,
        weight_decay=0.05,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        bf16=True,
        logging_steps=max(10, eval_steps_eff // 4),
        eval_strategy="steps",
        eval_steps=eval_steps_eff,
        save_strategy="steps",
        save_steps=eval_steps_eff,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="val_mmarco_cosine_ndcg@10",
        greater_is_better=True,
        report_to="none",
        seed=42,
        data_seed=42,
        dataloader_drop_last=True,
        gradient_checkpointing=is_llm,
        gradient_checkpointing_kwargs={"use_reentrant": False} if is_llm else {},
    )

    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=ds,
        loss=loss,
        evaluator=evaluator,
    )
    trainer.train()

    final_path = f"models/{run_name}_final"
    os.makedirs(final_path, exist_ok=True)

    if is_llm:
        model[0].auto_model.config.use_cache = True

    if use_lora:
        model[0].auto_model = model[0].auto_model.merge_and_unload()
        
    model.save_pretrained(final_path)
    print(f"saved -> {final_path}")

    with open(os.path.join(final_path, "retriever_meta.json"), "w", encoding="utf-8") as f:
        json.dump({
            "is_e5": is_e5,
            "is_llm": is_llm,
            "base_model": model_id,
            "use_lora": use_lora,
            "lora_rank": lora_rank if use_lora else None,
            "lora_alpha": lora_alpha if use_lora else None,
        }, f, ensure_ascii=False, indent=2)

    eval_metrics = {}
    for log in reversed(trainer.state.log_history):
        if any(k.startswith("eval_") for k in log):
            eval_metrics = {k: v for k, v in log.items() if k.startswith("eval_")}
            break

    os.makedirs("data/processed/metrics", exist_ok=True)
    with open(f"data/processed/metrics/train_{run_name}.json", "w", encoding="utf-8") as f:
        json.dump({
            "run_name": run_name,
            "model_id": model_id,
            "use_lora": use_lora,
            "lora_rank": lora_rank if use_lora else None,
            "lora_alpha": lora_alpha if use_lora else None,
            "lr": lr,
            "epochs": epochs,
            "batch_size": batch_size,
            "grad_accum": grad_accum,
            "max_seq_length": max_seq_length,
            "eval_metrics": eval_metrics,
        }, f, ensure_ascii=False, indent=2)

    del model, trainer, loss
    gc.collect()
    torch.cuda.empty_cache()

## Эксперименты

In [ ]:
BASE_MODEL = "intfloat/multilingual-e5-base"
LLM_MODEL = "Qwen/Qwen3-1.7B"

# encoder-only: full vs LoRA
run_experiment(BASE_MODEL, "e5_full_ft", use_lora=False,
    lr=2e-5, batch_size=64, max_seq_length=512, epochs=1)
run_experiment(BASE_MODEL, "e5_lora_r16", use_lora=True,
    lora_rank=16, lora_alpha=32, lr=1e-4,
    batch_size=64, max_seq_length=512, epochs=2)
run_experiment(BASE_MODEL, "e5_lora_r32", use_lora=True,
    lora_rank=32, lora_alpha=64, lr=1e-4,
    batch_size=64, max_seq_length=512, epochs=2)

# аблейшен: без instruction negatives
run_experiment(BASE_MODEL, "e5_lora_r16_no_instneg", use_lora=True,
    lora_rank=16, lora_alpha=32, lr=1e-4,
    batch_size=64, max_seq_length=512, epochs=2,
    use_inst_neg=False)

# LLM-бэкбон (last-token pooling)
run_experiment(LLM_MODEL, "qwen_lora_r16", use_lora=True,
    lora_rank=16, lora_alpha=32, lr=1e-4,
    batch_size=32, max_seq_length=512, epochs=1)

# аблейшены LoRA alpha при фикс r=16
run_experiment(BASE_MODEL, "e5_lora_r16_a16", use_lora=True,
    lora_rank=16, lora_alpha=16, lr=1e-4,
    batch_size=64, max_seq_length=512, epochs=2)
run_experiment(BASE_MODEL, "e5_lora_r16_a64", use_lora=True,
    lora_rank=16, lora_alpha=64, lr=1e-4,
    batch_size=64, max_seq_length=512, epochs=2)

# аблейшен learning rate
run_experiment(BASE_MODEL, "e5_lora_r16_lr2e5", use_lora=True,
    lora_rank=16, lora_alpha=32, lr=2e-5,
    batch_size=64, max_seq_length=512, epochs=2)
run_experiment(BASE_MODEL, "e5_lora_r16_lr3e4", use_lora=True,
    lora_rank=16, lora_alpha=32, lr=3e-4,
    batch_size=64, max_seq_length=512, epochs=2)

# другая base-модель
run_experiment("BAAI/bge-m3", "bge_m3_lora_r16", use_lora=True,
    lora_rank=16, lora_alpha=32, lr=1e-4,
    batch_size=32, max_seq_length=512, epochs=2)

# аблейшен epochs
run_experiment(BASE_MODEL, "e5_lora_r16_a64_1ep", use_lora=True,
    lora_rank=16, lora_alpha=64, lr=1e-4,
    batch_size=64, max_seq_length=512, epochs=1)
run_experiment(BASE_MODEL, "e5_lora_r16_a64_3ep", use_lora=True,
    lora_rank=16, lora_alpha=64, lr=1e-4,
    batch_size=64, max_seq_length=512, epochs=3)

# аблейшен ранга при alpha=64
run_experiment(BASE_MODEL, "e5_lora_r8_a64", use_lora=True,
    lora_rank=8, lora_alpha=64, lr=1e-4,
    batch_size=64, max_seq_length=512, epochs=2)
run_experiment(BASE_MODEL, "e5_lora_r32_a64", use_lora=True,
    lora_rank=32, lora_alpha=64, lr=1e-4,
    batch_size=64, max_seq_length=512, epochs=2)

# аблейшен размера трейна
for n in [5000, 15000]:
    run_experiment(BASE_MODEL, f"e5_lora_r16_n{n // 1000}", use_lora=True,
            lora_rank=16, lora_alpha=32, lr=1e-4,
            batch_size=64, max_seq_length=512, epochs=2,
            train_data_subset=train_data[:n])

# Оценка на mMARCO / MS MARCO dev
Семплируем ~1М документов из корпуса (reservoir) + добавляем все qrels-документы.

In [ ]:
def evaluate_dataset(dataset_name, models_to_eval,
                     max_q=1000, corpus_size=1_000_000,
                     top_bm25=1000, K=10, seed=42):
    print(f"\n=== {dataset_name} | max_q={max_q} | corpus={corpus_size} ===")
    ds = ir_datasets.load(dataset_name)

    queries_dict = {}
    for q in ds.queries_iter():
        queries_dict[str(q.query_id)] = q.text
        if len(queries_dict) >= max_q:
            break
    qids = list(queries_dict.keys())
    qids_set = set(qids)

    qrels = defaultdict(set)
    for qr in ds.qrels_iter():
        if str(qr.query_id) in qids_set and int(qr.relevance) > 0:
            qrels[str(qr.query_id)].add(str(qr.doc_id))

    cand_per_q = defaultdict(list)
    for sd in ds.scoreddocs_iter():
        qid = str(sd.query_id)
        if qid not in qids_set:
            continue
        if len(cand_per_q[qid]) >= top_bm25:
            continue
        cand_per_q[qid].append(str(sd.doc_id))

    needed_qrels = set().union(*qrels.values())
    rng = random.Random(seed)
    sample_ids = [None] * corpus_size
    sample_texts = [None] * corpus_size
    qrels_docs = {}
    total_seen = 0

    # reservoir sampling за один проход по корпусу
    for d in tqdm(ds.docs_iter()):
        did = str(d.doc_id)
        text = d.text[:2000]
        if did in needed_qrels:
            qrels_docs[did] = text
        if total_seen < corpus_size:
            sample_ids[total_seen] = did
            sample_texts[total_seen] = text
        else:
            j = rng.randint(0, total_seen)
            if j < corpus_size:
                sample_ids[j] = did
                sample_texts[j] = text
        total_seen += 1

    print(f"docs seen: {total_seen} | qrels: {len(qrels_docs)}/{len(needed_qrels)}")

    corpus = {did: t for did, t in zip(sample_ids, sample_texts) if did is not None}
    corpus.update(qrels_docs)
    doc_ids = list(corpus.keys())
    raw_doc_texts = [corpus[d] for d in doc_ids]
    print(f"corpus: {len(doc_ids)}")

    def metrics(ranked, rel):
        out = {f"Hit@{k}": 0 for k in (1, 3, 5, 10)}
        out.update({"MRR@10": 0.0, "MAP@10": 0.0, "Recall@10": 0.0, "NDCG@10": 0.0})
        if not rel:
            return out
        hits, dcg = 0, 0.0
        for r, d in enumerate(ranked[:K], 1):
            if d in rel:
                hits += 1
                for k in (1, 3, 5, 10):
                    if r <= k:
                        out[f"Hit@{k}"] = 1
                if out["MRR@10"] == 0.0:
                    out["MRR@10"] = 1.0 / r
                out["MAP@10"] += hits / r
                dcg += 1.0 / np.log2(r + 1)
        idcg = sum(1.0 / np.log2(i + 1) for i in range(1, min(len(rel), K) + 1))
        out["NDCG@10"] = dcg / idcg if idcg > 0 else 0.0
        out["Recall@10"] = hits / len(rel)
        out["MAP@10"] /= min(len(rel), K)
        return out

    results = {}

    print("[BM25]")
    agg = defaultdict(list)
    for qid in qids:
        for k, v in metrics(cand_per_q[qid], qrels[qid]).items():
            agg[k].append(v)
    results["BM25"] = {k: float(np.mean(v)) for k, v in agg.items()}

    for name, path in models_to_eval.items():
        print(f"\n[{name}]")
        m, is_e5, _ = load_st_model(path)
        qp = "query: " if is_e5 else ""
        dp = "passage: " if is_e5 else ""

        d_emb = m.encode([dp + t for t in raw_doc_texts], batch_size=256,
                         normalize_embeddings=True, convert_to_numpy=True,
                         show_progress_bar=True).astype("float32")
        q_emb = m.encode([qp + queries_dict[qid] for qid in qids], batch_size=256,
                         normalize_embeddings=True, convert_to_numpy=True,
                         show_progress_bar=True).astype("float32")

        index = faiss.IndexFlatIP(d_emb.shape[1])
        index.add(d_emb)
        _, I = index.search(q_emb, K)

        agg = defaultdict(list)
        for i, qid in enumerate(qids):
            ranked = [doc_ids[idx] for idx in I[i]]
            for k, v in metrics(ranked, qrels[qid]).items():
                agg[k].append(v)
        results[name] = {k: float(np.mean(v)) for k, v in agg.items()}

        del m, d_emb, q_emb, index
        gc.collect()
        torch.cuda.empty_cache()

    return results


models_dict = {
    "e5-base (Baseline)": "intfloat/multilingual-e5-base",
    "bge-m3 (Baseline)": "BAAI/bge-m3",
    "e5_full_ft": "models/e5_full_ft_final",
    "e5_lora_r16": "models/e5_lora_r16_final",
    "e5_lora_r32": "models/e5_lora_r32_final",
    "e5_lora_r16_no_instneg": "models/e5_lora_r16_no_instneg_final",
    "qwen_lora_r16": "models/qwen_lora_r16_final",
    "e5_lora_r16_a16": "models/e5_lora_r16_a16_final",
    "e5_lora_r16_a64": "models/e5_lora_r16_a64_final",
    "e5_lora_r16_lr2e5": "models/e5_lora_r16_lr2e5_final",
    "e5_lora_r16_lr3e4": "models/e5_lora_r16_lr3e4_final",
    "bge_m3_lora_r16": "models/bge_m3_lora_r16_final",
    "e5_lora_r16_a64_1ep": "models/e5_lora_r16_a64_1ep_final",
    "e5_lora_r16_a64_3ep": "models/e5_lora_r16_a64_3ep_final",
    "e5_lora_r8_a64": "models/e5_lora_r8_a64_final",
    "e5_lora_r32_a64": "models/e5_lora_r32_a64_final",
    "e5_lora_r16_n5": "models/e5_lora_r16_n5_final",
    "e5_lora_r16_n15": "models/e5_lora_r16_n15_final",
}

res_ru = evaluate_dataset("mmarco/ru/dev/small", models_dict,
                         max_q=1000, corpus_size=1_000_000)
res_en = evaluate_dataset("msmarco-passage/dev/small", models_dict,
                         max_q=1000, corpus_size=1_000_000)

os.makedirs("data/processed/metrics", exist_ok=True)
with open("data/processed/metrics/dev_ru_results.json", "w", encoding="utf-8") as f:
    json.dump(res_ru, f, ensure_ascii=False, indent=2)
with open("data/processed/metrics/dev_en_results.json", "w", encoding="utf-8") as f:
    json.dump(res_en, f, ensure_ascii=False, indent=2)

# Оценка на golden set
Глобальный пул всех golden-документов. Считаем MRR без инструкции,
с инструкцией и с инструкцией-шумом (от другого запроса), плюс p-MRR.

In [ ]:
def evaluate_golden_set(models_to_eval,
                        golden_path="data/golden_set/golden_for_manual_review.json",
                        seed=42):
    with open(golden_path, encoding="utf-8") as f:
        golden = json.load(f)

    def get_neg(x):
        v = x["instruction_negative"]
        return v["passage"] if isinstance(v, dict) else v

    pool = []
    for qi, x in enumerate(golden):
        pool.append(("pos", qi, x["positive_text"]))
        pool.append(("instr_neg", qi, get_neg(x)))
        for j, n in enumerate(x.get("hard_negatives", [])):
            pool.append((f"hard_{j}", qi, n))
        for j, n in enumerate(x.get("random_negatives", [])):
            pool.append((f"rand_{j}", qi, n))
    pos_idx_of = {qi: i for i, (kind, qi, _) in enumerate(pool) if kind == "pos"}
    print(f"Golden pool: {len(pool)} docs, {len(golden)} queries")

    rng = random.Random(seed)
    n = len(golden)
    noise_instr = []
    for i in range(n):
        j = rng.randrange(n - 1)
        if j >= i:
            j += 1
        noise_instr.append(golden[j]["instruction"])

    def rank_of_pos(scores, pi):
        return 1 + int((scores > scores[pi]).sum())

    def agg_metrics(ranks_no, ranks_wi, ranks_nz):
        mrr_no = float(np.mean([1 / r for r in ranks_no]))
        mrr_wi = float(np.mean([1 / r for r in ranks_wi]))
        mrr_nz = float(np.mean([1 / r for r in ranks_nz]))
        pmrr_list = [1 - rn / ro if rn <= ro else ro / rn - 1 for ro, rn in zip(ranks_no, ranks_wi)]
        helped = float(np.mean([rn < ro for ro, rn in zip(ranks_no, ranks_wi)]))
        hurt = float(np.mean([rn > ro for ro, rn in zip(ranks_no, ranks_wi)]))
        return {
            "MRR (No Inst)": mrr_no,
            "MRR (With Inst)": mrr_wi,
            "MRR (Noise Inst)": mrr_nz,
            "p-MRR": float(np.mean(pmrr_list)),
            "% helped": helped,
            "% hurt": hurt,
        }

    results = {}

    print("[BM25]")

    def tok(s):
        return re.findall(r"\w+", s.lower())

    bm25 = BM25Okapi([tok(t) for _, _, t in pool])
    r_no, r_wi, r_nz = [], [], []
    for i, x in enumerate(golden):
        pi = pos_idx_of[i]
        r_no.append(rank_of_pos(bm25.get_scores(tok(x["query"])), pi))
        r_wi.append(rank_of_pos(bm25.get_scores(tok(x["instruction"] + " " + x["query"])), pi))
        r_nz.append(rank_of_pos(bm25.get_scores(tok(noise_instr[i] + " " + x["query"])), pi))
    results["BM25"] = agg_metrics(r_no, r_wi, r_nz)

    for name, path in models_to_eval.items():
        print(f"\n[{name}]")
        m, is_e5, _ = load_st_model(path)
        qp = "query: " if is_e5 else ""
        dp = "passage: " if is_e5 else ""

        d_emb = m.encode([dp + t for _, _, t in pool], batch_size=128,
                         normalize_embeddings=True, convert_to_numpy=True,
                         show_progress_bar=False)
        q_no = m.encode([qp + x["query"] for x in golden],
                        normalize_embeddings=True, convert_to_numpy=True,
                        show_progress_bar=False)
        q_wi = m.encode([qp + x["instruction"] + " " + x["query"] for x in golden],
                        normalize_embeddings=True, convert_to_numpy=True,
                        show_progress_bar=False)
        q_nz = m.encode([qp + noise_instr[i] + " " + x["query"] for i, x in enumerate(golden)],
                        normalize_embeddings=True, convert_to_numpy=True,
                        show_progress_bar=False)

        r_no, r_wi, r_nz = [], [], []
        for i in range(len(golden)):
            pi = pos_idx_of[i]
            r_no.append(rank_of_pos(d_emb @ q_no[i], pi))
            r_wi.append(rank_of_pos(d_emb @ q_wi[i], pi))
            r_nz.append(rank_of_pos(d_emb @ q_nz[i], pi))
        results[name] = agg_metrics(r_no, r_wi, r_nz)

        del m, d_emb, q_no, q_wi, q_nz
        gc.collect()
        torch.cuda.empty_cache()

    return results


res_golden = evaluate_golden_set(models_dict)
print("\n--- Golden set ---")
print(pd.DataFrame(res_golden).T)

with open("data/processed/metrics/res_golden.json", "w", encoding="utf-8") as f:
    json.dump(res_golden, f, ensure_ascii=False, indent=2)